# Parte 1: Resumo do Fluxo de Trabalho (Scalp EEG)

Este é um resumo consolidado das etapas essenciais para carregar e limpar um EEG de couro cabeludo (Scalp EEG), padronizando o fluxo visto nos notebooks anteriores.

In [29]:
import matplotlib
import pathlib
import numpy as np
import mne
import mne_bids

# Configuração do backend gráfico
try:
    matplotlib.use('Qt5Agg')
except:
    matplotlib.use('Agg') # Fallback se Qt não estiver disponível

## 1.1 Carregamento e Pré-processamento Básico

A função abaixo encapsula as correções iniciais necessárias para a maioria dos arquivos EDF brutos (conversão de unidade, remoção de offset DC, re-referência e filtro de linha de base).

In [30]:
def preprocess_standard_eeg(raw):
    """
    Pipeline padrão para limpeza inicial de EEG de superfície.
    """
    raw.load_data()
    data = raw.get_data()
    
    # 1. Correção de Unidade (µV para Volts)
    if np.max(np.abs(data)) > 1e-3: 
        print("[INFO] Convertendo de µV para Volts...")
        raw.apply_function(lambda x: x / 1e6)
        
    # 2. Remover Offset DC (Centralizar o sinal)
    print("[INFO] Removendo offset DC (média do canal)...")
    raw.apply_function(lambda x: x - x.mean())
    
    # 3. Re-referência (Média Comum)
    print("[INFO] Aplicando referência média (CAR)...")
    raw.set_eeg_reference('average', projection=False)
    
    # 4. Filtro Passa-Alta (High-pass) para remover drift
    print("[INFO] Aplicando filtro passa-alta de 1.0 Hz...")
    raw.filter(l_freq=1.0, h_freq=None)
    
    return raw

# Exemplo de uso (caminho hipotético baseado no seu anterior)
fname = "./dados_hands/sub-02_task-MIvsRest_run-2_eeg.edf"
raw = mne.io.read_raw_edf(fname, preload=True)
raw_clean = preprocess_standard_eeg(raw)
raw_clean.plot(title="EEG Processado")

Extracting EDF parameters from c:\Users\danil\Downloads\mne-eeg-python-introducao-main\mne-eeg-python-introducao-main\dados_hands\sub-02_task-MIvsRest_run-2_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 54624  =      0.000 ...   436.992 secs...
[INFO] Convertendo de µV para Volts...
[INFO] Removendo offset DC (média do canal)...
[INFO] Aplicando referência média (CAR)...
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
[INFO] Aplicando filtro passa-alta de 1.0 Hz...
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 H

C:\Users\danil\AppData\Local\Temp\ipykernel_15716\3672385389.py:29: RuntimeWarning: Omitted 1 annotation(s) that were outside data range.
  raw = mne.io.read_raw_edf(fname, preload=True)


Channels marked as bad:
none


---
# Parte 2: Análise de Epilepsia (iEEG/BIDS)

Agora vamos focar na estrutura de pastas `sub-RESP0059/ses-SITUATION1A/ieeg` indica que este é um dataset **BIDS** de **iEEG (EEG Intracraniano)**.

**Diferenças Chave do iEEG:**
1.  **Sinal:** Muito mais limpo e com amplitude maior que o de superfície.
2.  **Frequência:** Interessam frequências muito mais altas (HFOs - High Frequency Oscillations), então não filtramos em 40Hz.
3.  **Artefatos:** Menos piscadas e músculos, mas muito ruído de linha (60Hz) e equipamentos hospitalares.

## 2.1 Lendo o Dataset BIDS
O caminho raiz é `dados_epilepsy_2/ds003844-download`.

In [36]:
# Caminho para o dataset BIDS (ajuste se necessário)
bids_root = pathlib.Path('dados_epilepsy_2/ds003844-download')

# Variável de controle para saber se temos dados reais
has_real_data = False

if not bids_root.exists():
    print(f"⚠️ AVISO: A pasta {bids_root} não foi encontrada.")
    print("Vamos pular o carregamento real e ir direto para a Simulação Didática.")
else:
    try:
        # Definindo o caminho BIDS específico
        bids_path = mne_bids.BIDSPath(
            subject='RESP0059',
            session='SITUATION1A',
            task='acute',
            datatype='ieeg', 
            root=bids_root
        )
        
        print(f"Lendo arquivo: {bids_path.basename}")
        # Ler os dados
        raw_ieeg = mne_bids.read_raw_bids(bids_path, verbose=False)
        raw_ieeg.load_data()
        has_real_data = True
        print("✅ Dados de iEEG carregados com sucesso!")
        print(raw_ieeg.info)
        for anotacao in raw_ieeg.annotations:
            print(f"Tempo: {anotacao['onset']:.2f}s | Label: {anotacao['description']}")
    except Exception as e:
        print(f"Erro ao ler BIDS: {e}")

Lendo arquivo: sub-RESP0059_ses-SITUATION1A_task-acute
Reading 0 ... 665586  =      0.000 ...   324.993 secs...


C:\Users\danil\AppData\Local\Temp\ipykernel_15716\3703222262.py:23: RuntimeWarning: No BIDS -> MNE mapping found for channel type "OTHER". Type of channel "Riv01" will be set to "misc".
  raw_ieeg = mne_bids.read_raw_bids(bids_path, verbose=False)
C:\Users\danil\AppData\Local\Temp\ipykernel_15716\3703222262.py:23: RuntimeWarning: No BIDS -> MNE mapping found for channel type "OTHER". Type of channel "Riv02" will be set to "misc".
  raw_ieeg = mne_bids.read_raw_bids(bids_path, verbose=False)
C:\Users\danil\AppData\Local\Temp\ipykernel_15716\3703222262.py:23: RuntimeWarning: No BIDS -> MNE mapping found for channel type "OTHER". Type of channel "Riv03" will be set to "misc".
  raw_ieeg = mne_bids.read_raw_bids(bids_path, verbose=False)
C:\Users\danil\AppData\Local\Temp\ipykernel_15716\3703222262.py:23: RuntimeWarning: No BIDS -> MNE mapping found for channel type "OTHER". Type of channel "Riv04" will be set to "misc".
  raw_ieeg = mne_bids.read_raw_bids(bids_path, verbose=False)
C:\Users

✅ Dados de iEEG carregados com sucesso!
<Info | 11 non-empty values
 bads: 6 items (Gr20, Riv01, Riv02, Riv03, Riv04, Riv05)
 ch_names: Gr16, Gr17, Gr18, Gr19, Gr20, Rst01, Rst02, Rst03, Gr11, Gr12, ...
 chs: 26 ECoG, 5 misc, 1 ECG
 custom_ref_applied: False
 dig: 29 items (3 Cardinal, 26 EEG)
 highpass: 0.0 Hz
 line_freq: 50.0
 lowpass: 1024.0 Hz
 meas_date: unspecified
 nchan: 32
 projs: []
 sfreq: 2048.0 Hz
 subject_info: <subject_info | his_id: sub-RESP0059, sex: 2>
>
Tempo: 0.03s | Label: artefact
Tempo: 0.98s | Label: good data segment
Tempo: 4.99s | Label: artefact
Tempo: 7.93s | Label: artefact
Tempo: 13.90s | Label: artefact
Tempo: 13.90s | Label: artefact
Tempo: 13.90s | Label: artefact
Tempo: 13.90s | Label: artefact
Tempo: 13.90s | Label: artefact
Tempo: 13.90s | Label: artefact
Tempo: 26.87s | Label: artefact
Tempo: 27.36s | Label: artefact
Tempo: 27.36s | Label: artefact
Tempo: 27.36s | Label: artefact
Tempo: 231.92s | Label: artefact
Tempo: 231.92s | Label: artefact
Temp

## 2.2 Filtragem para Epilepsia

Em epilepsia, **NÃO** usamos o filtro passa-baixa padrão de 40Hz ou 50Hz, pois as "Pontas" (Spikes) epilépticas e oscilações de alta frequência (HFOs) são rápidas.

**Estratégia:**
1.  **Notch Filter:** Remover ruído da tomada (60Hz no Brasil/EUA, 50Hz na Europa).
2.  **Passa-Banda Largo:** Manter de 1Hz até 100Hz (ou mais) para ver as descargas epilépticas claras.

In [34]:
if has_real_data:
    print("\n--- APLICANDO FILTROS DE EPILEPSIA ---")
    
    # 1. Filtro Notch (Remover ruído da rede elétrica - 60Hz e harmônicos)
    # Fundamental em iEEG pois os cabos funcionam como antenas
    freqs_notch = np.arange(60, 241, 60)
    print(f"1. Aplicando Notch em: {freqs_notch} Hz (Limpando ruído de linha)")
    raw_ieeg.notch_filter(freqs=freqs_notch, verbose=False)

    # 2. Filtro Passa-Banda Largo (1Hz - 100Hz ou mais)
    # POR QUÊ? 
    # - Low freq (1Hz): Remove oscilação lenta (suor/movimento).
    # - High freq (100Hz): Mantemos frequências altas para ver o "Spike" (a ponta aguda).
    # Se usarmos 40Hz (padrão de sono), a ponta epiléptica é arredondada e perdemos o diagnóstico.
    print("2. Aplicando Bandpass (1Hz - 100Hz) para preservar Spikes...")
    raw_ieeg.filter(l_freq=1.0, h_freq=100.0, verbose=False)
    
    print("✅ Dados filtrados e prontos.")


--- APLICANDO FILTROS DE EPILEPSIA ---
1. Aplicando Notch em: [ 60 120 180 240] Hz (Limpando ruído de linha)
2. Aplicando Bandpass (1Hz - 100Hz) para preservar Spikes...
✅ Dados filtrados e prontos.


### Célula 4: ICA em Epilepsia (O Grande Perigo)
*Uma breve demonstração teórica.*

In [ ]:
if has_real_data:
    print("\n--- PROCESSAMENTO ICA ---")
    
    # 1. Configurar e Calcular o ICA
    ica = mne.preprocessing.ICA(
        n_components=15, 
        method='picard', 
        random_state=42
    )
    print("Calculando componentes independentes (pode demorar)...")
    ica.fit(raw_ieeg, verbose=False)
    
    # 2. Mostrar os componentes (para você decidir o que é lixo)
    # O parâmetro show=False evita travar o script
    ica.plot_sources(raw_ieeg, title="Componentes ICA (Identifique os ruídos)", show=False)
    
    # 3. Definir o que será removido
    # (Aqui estou forçando 0 e 1 como exemplo. Na vida real, você olha o gráfico acima e decide)
    ica.exclude = [0, 1] 
    print(f"Componentes marcados para exclusão: {ica.exclude}")
    
    # 4. APLICAR A LIMPEZA (O passo crucial)
    print("Aplicando a limpeza no sinal...")
    # Criamos uma cópia para preservar o original
    raw_clean = raw_ieeg.copy()
    ica.apply(raw_clean)
    
    # 5. PLOTAR O SINAL LIMPO
    print("✅ Plotando o sinal final PÓS-ICA:")
    raw_clean.plot(
        title="Sinal Limpo (Após remoção dos componentes ICA)", 
        duration=10, 
        n_channels=20, 
        scalings='auto'
    )
    
    # (Opcional) Plotar lado a lado para comparar (Overlay)
    # ica.plot_overlay(raw_ieeg, exclude=[0, 1], picks='eeg')


--- PROCESSAMENTO ICA ---
Calculando componentes independentes (pode demorar)...
Creating RawArray with float64 data, n_channels=16, n_times=665587
    Range : 0 ... 665586 =      0.000 ...   324.993 secs
Ready.
Componentes marcados para exclusão: [0, 1]
Aplicando a limpeza no sinal...
Applying ICA to Raw instance
    Transforming to ICA space (15 components)
    Zeroing out 2 ICA components
    Projecting back using 25 PCA components
✅ Plotando o sinal final PÓS-ICA:


Channels marked as bad:
['Gr20', 'Riv01', 'Riv02', 'Riv03', 'Riv04', 'Riv05']


## 2.4 Padrões de Epilepsia: Teoria

Na análise visual ou automática, buscamos principalmente:

1.  **IEDs (Descargas Interictais):** Ocorrem *entre* as crises.
    * **Spike (Ponta):** Uma onda muito aguda e rápida (< 70ms).
    * **Sharp Wave (Onda Aguda):** Um pouco mais lenta que o spike (70-200ms).
    * **Spike-and-Wave (Ponta-Onda):** Um spike seguido imediatamente de uma onda lenta.

2.  **Ictal (Crise):** Uma mudança abrupta no ritmo.
    * Geralmente começa com uma atividade rápida de baixa voltagem e evolui para ondas rítmicas de alta amplitude que diminuem de frequência até parar.

### Simulando Padrões Epilépticos

Como não podemos garantir que o arquivo lido tenha uma crise exatamente no trecho carregado, vamos **simular** como esses dados aparecem no MNE para você aprender a identificar.

In [ ]:
print("\n--- GERANDO VISUALIZAÇÃO CLÍNICA (SIMULAÇÃO) ---")

# --- 1. CONFIGURAÇÃO DA SIMULAÇÃO ---
sfreq = 250  
duration = 12 # segundos
n_channels = 3 
times = np.arange(0, duration, 1/sfreq)

# Gerar ruído de fundo (Atividade cerebral basal)
rng = np.random.default_rng(42)
data = rng.normal(loc=0, scale=3e-6, size=(n_channels, len(times))) 

# Função para desenhar a morfologia "Ponta-Onda"
def add_event(data_arr, t_idx, ch_idx, scale=1.0):
    # Amplitude em Volts (120 uV)
    spike_amp = 150e-6 * scale
    wave_amp = 90e-6 * scale
    
    # Spike (Triângulo agudo - Excitação)
    data_arr[ch_idx, t_idx : t_idx+4] += np.linspace(0, spike_amp, 4)
    data_arr[ch_idx, t_idx+4 : t_idx+8] -= np.linspace(0, spike_amp, 4)
    
    # Wave (Senoide lenta - Inibição)
    wave_len = 40
    wave_shape = np.sin(np.linspace(0, np.pi, wave_len)) * wave_amp
    data_arr[ch_idx, t_idx+8 : t_idx+8+wave_len] += wave_shape

# Injetar Crises nos segundos 3.0 e 7.5
events_times = [3.0, 7.5]
for t in events_times:
    idx = int(t * sfreq)
    add_event(data, idx, 0, scale=1.0) # Foco Principal
    add_event(data, idx, 1, scale=0.5) # Propagação menor
    # Canal 2 é saudável (só ruído)

# Converter para µV para plotagem ficar legível
data_uv = data * 1e6

# --- 2. VISUALIZAÇÃO "ARTIGO CIENTÍFICO" ---

# Cores e Estilo
C_FOCO = "#FF6B6B"  # Salmão (Foco)
C_VIZ  = "#FFD93D"  # Amarelo (Propagação)
C_NORM = "#4D96FF"  # Azul (Normal)
C_BG   = "#F9F9F9"  # Fundo leve

fig = plt.figure(figsize=(14, 9), facecolor=C_BG)
plt.subplots_adjust(hspace=0.3)

# --- PAINEL A: Visão Geral (Estilo Clínico com Offset) ---
ax_main = plt.subplot2grid((3, 2), (0, 0), colspan=2, rowspan=2)
ax_main.set_facecolor('white')

# Configurar Offset para empilhar as linhas (como no monitor do hospital)
offset = 250 # uV de distância entre canais
y_pos = [offset*2, offset, 0]
labels = ["Foco (Canal 1)", "Propagação (Canal 2)", "Controle (Canal 3)"]
colors = [C_FOCO, C_VIZ, C_NORM]

for i in range(3):
    # Plota somando o offset para separar as linhas verticalmente
    ax_main.plot(times, data_uv[i] + y_pos[i], color=colors[i], lw=1.2)
    # Nome do canal na esquerda
    ax_main.text(-0.5, y_pos[i], labels[i], color=colors[i], fontweight='bold', ha='right', va='center')

# Marcar onde as crises ocorreram
for t in events_times:
    # Faixa vertical colorida
    ax_main.axvspan(t-0.2, t+0.5, color=C_FOCO, alpha=0.1, lw=0)
    ax_main.text(t, y_pos[0]+120, "CRISE", color=C_FOCO, fontweight='bold', ha='center')

# Limpeza visual
ax_main.set_yticks([])
ax_main.spines['top'].set_visible(False)
ax_main.spines['right'].set_visible(False)
ax_main.spines['left'].set_visible(False)
ax_main.set_title("A. Monitoramento EEG: Detecção de Eventos", loc='left', fontsize=14, fontweight='bold', color='#333333')
ax_main.set_xlabel("Tempo (segundos)")


# --- PAINEL B: Zoom na Morfologia (Didático) ---
ax_zoom = plt.subplot2grid((3, 2), (2, 0), colspan=2)
ax_zoom.set_facecolor('white')

# Recortar o primeiro evento para zoom (2.9s a 3.3s)
start_z = int(2.9 * sfreq)
end_z = int(3.3 * sfreq)
t_zoom = times[start_z:end_z]
d_zoom = data_uv[0, start_z:end_z] # Apenas canal foco

ax_zoom.plot(t_zoom, d_zoom, color=C_FOCO, lw=2.5)

# Anotações Explicativas
# 1. Seta para o Spike
ax_zoom.annotate('Spike (Ponta)\nDescarga Rápida', 
                 xy=(3.015, 150), xytext=(2.95, 120),
                 arrowprops=dict(facecolor=C_FOCO, shrink=0.05),
                 color=C_FOCO, fontweight='bold', ha='center')

# 2. Seta para a Wave
ax_zoom.annotate('Wave (Onda Lenta)\nInibição Pós-Disparo', 
                 xy=(3.10, 80), xytext=(3.20, 100),
                 arrowprops=dict(facecolor=C_NORM, shrink=0.05),
                 color=C_NORM, fontweight='bold', ha='center')

ax_zoom.set_title("B. Detalhe Morfológico: Padrão Ponta-Onda", loc='left', fontsize=12, fontweight='bold', color='#555555')
ax_zoom.set_ylabel("Amplitude (µV)")
ax_zoom.set_xlabel("Tempo Detalhado (s)")
ax_zoom.grid(True, linestyle='--', alpha=0.3)
ax_zoom.spines['top'].set_visible(False)
ax_zoom.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()